### DASPack Demo

Author: Benz Poobua (spoobua@stanford.edu)

Reference: Seguí, A., Ugalde, A., Fichtner, A., Ventosa, S., & Morros, J. R. (2025). DASPack: Controlled Data Compression for Distributed Acoustic Sensing. Geophysical Journal International.
DOI: 10.1093/gji/ggaf397

DASPack: https://github.com/asleix/daspack

In [1]:
import os 
import sys
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio

from daspack import DASCoder, Quantizer

### Offshore DAS Dataset (Strain)
Reference: Williams, E. F., Fernandez-Ruiz, M. R., Magalhaes, R., Vanthillo, R., Zhan, Z., Gonzalez-Herraez, M., & Martins, H. F. (2019). Belgium Distributed Acoustic Sensing Array Raw Data (1.0) [Data set]. CaltechDATA. https://doi.org/10.22002/D1.1296

In [2]:
# Define paths
data_dir = os.path.join('..', 'data', 'raw_offshore')
input_filename = 'mat_2018_08_19_00h28m05s_Parkwind_HDAS_2Dmap_StrainData_2D.mat'
mat_path = os.path.join(data_dir, input_filename)

output_filename = input_filename.replace('.mat', '_compressed.npz')
out_path = os.path.join(data_dir, output_filename)

In [3]:
# Load data
mat_contents = sio.loadmat(mat_path)

# Print the keys 
print("Keys in the file:", mat_contents.keys())

# Extract the data matrix
strain_data = mat_contents['Data_2D'] # Original float64 array
print("Shape of the data matrix:", strain_data.shape)

Keys in the file: dict_keys(['__header__', '__version__', '__globals__', 'Data_2D'])
Shape of the data matrix: (4192, 42000)


In [4]:
# Define metadata parameters
dx = 10.0 # Channel spacing in meters
fs = 10.0 # Sampling frequency in Hz
dt = 1.0 / fs # Sampling interval in seconds

num_channels, num_samples = strain_data.shape

# Calculate total time and distance
total_time_sec = num_samples / fs
total_distance_km = (num_channels * dx) / 1000.0
print(f"Total time: {total_time_sec:.2f} seconds")
print(f"Total distance: {total_distance_km:.2f} km")

Total time: 4200.00 seconds
Total distance: 41.92 km


In [5]:
# Create the axes for plotting
t_axis = np.arange(strain_data.shape[1]) / fs
x_axis = np.arange(strain_data.shape[0]) * dx / 1000.0  # Convert meters to km

das_data = {
    'data': strain_data,
    't_axis': t_axis,
    'x_axis': x_axis
}

In [6]:
# Scale factor: 1e5 preserves 5 decimal places and fits 4286.5 into int32
scale_factor = 1e5 

# Quantize the data to int32 
strain_data_scaled = np.round(strain_data * scale_factor)
strain_data_int = np.ascontiguousarray(strain_data_scaled, dtype=np.int32)

In [7]:
# Compress
coder = DASCoder(threads=4)
comp_bytes = coder.encode(strain_data_int, Quantizer.Lossless())
comp_array = np.frombuffer(comp_bytes, dtype=np.uint8)

# Save
np.savez(out_path, 
         compressed_das=comp_array, 
         original_shape=strain_data_int.shape,
         scale_factor=scale_factor,
         t_axis=t_axis,
         x_axis=x_axis,
         fs=fs,
         dx=dx)

DASPack: initialized global Rayon pool with 4 thread(s).


In [8]:
# Decode
decoded_bytes = comp_array.tobytes()
restored_int = coder.decode(decoded_bytes).reshape(strain_data_int.shape)
restored_float = restored_int.astype(np.float64) / scale_factor

In [9]:
# Error analysis
diff = strain_data - restored_float
max_err = np.max(np.abs(diff))
frob_err = np.linalg.norm(diff, ord='fro')
frob_orig = np.linalg.norm(strain_data, ord='fro')
rel_frob_err = frob_err / frob_orig

# File Sizes
raw_size_mb = os.path.getsize(mat_path) / (1024**2)
comp_size_mb = os.path.getsize(out_path) / (1024**2)

print(f"Original size: {raw_size_mb:.2f} MB")
print(f"Compressed size: {comp_size_mb:.2f} MB")
print(f"Max absolute error: {max_err:.2e}")
print(f"Frobenius norm of error: {frob_err:.2e}")
print(f"Relative Frobenius error: {rel_frob_err:.2e}")

Original size: 1281.24 MB
Compressed size: 460.93 MB
Max absolute error: 5.00e-06
Frobenius norm of error: 3.83e-02
Relative Frobenius error: 1.02e-08


In [10]:
# Load the compressed
compressed_path = os.path.join(data_dir, 'mat_2018_08_19_00h28m05s_Parkwind_HDAS_2Dmap_StrainData_2D_compressed.npz')
with np.load(compressed_path) as loader:
    comp_array = loader['compressed_das']
    orig_shape = tuple(loader['original_shape'])
    scale_factor = loader['scale_factor']

    t_axis = loader['t_axis']
    x_axis = loader['x_axis']

# Decode 
coder = DASCoder(threads=4)

# Convert the unit8 array back to bytes and decode
restored_int = coder.decode(comp_array.tobytes()).reshape(orig_shape)

# Restore to original float values
# Cast to float32 or float64
strain_data_restored = restored_int.astype(np.float64) / scale_factor

---

### Urban DAS Dataset (Strain Rate)
Reference: Stanford-2 Experimemt (Sand Hill Road)

In [11]:
data_dir = os.path.join('..', 'data', 'preprocessed', '20210901')
input_filename = '20210901_000000.npz'
in_path = os.path.join(data_dir, input_filename)

out_dir = os.path.join('..', 'data', 'preprocessed', '20210901_compressed')
os.makedirs(out_dir, exist_ok=True)
output_filename = input_filename.replace('.npz', '_comp.npz')
out_path = os.path.join(out_dir, output_filename)

In [12]:
# Load data 
with np.load(in_path) as ld:
    data_raw_orig = ld['data'][:] 

# Dynamic scaling factor to fit the data into int32 range
max_abs = np.max(np.abs(data_raw_orig))
scale_factor = min(1e7, np.floor(2e9 / (max_abs + 1e-9)))

# Multiply, round to nearest integer, and cast to contiguous int32
data_scaled_int = np.ascontiguousarray(np.round(data_raw_orig * scale_factor), dtype=np.int32)

# Compress
coder = DASCoder(threads=4)
comp_bytes = coder.encode(data_scaled_int, Quantizer.Lossless())
comp_array = np.frombuffer(comp_bytes, dtype=np.uint8)

# Save 
np.savez(out_path, 
         compressed_das=comp_array, 
         original_shape=data_scaled_int.shape,
         scale_factor=scale_factor)  # Must save the scale factor for decoding later

In [13]:
# Decode 
data_restored = coder.decode(comp_bytes).reshape(data_scaled_int.shape)
data_restored_float = data_restored.astype(np.float64) / scale_factor

In [14]:
# Error analysis 
error_diff = data_raw_orig - data_restored_float
error_max = np.max(np.abs(error_diff))
error_frob_norm = np.linalg.norm(error_diff, ord='fro')
orig_frob_norm = np.linalg.norm(data_raw_orig, ord='fro')
error_rel_frob = error_frob_norm / orig_frob_norm

# File Sizes 
size_original_mb = os.path.getsize(in_path) / (1024**2)
size_compressed_mb = os.path.getsize(out_path) / (1024**2)

print(f"Original size: {size_original_mb:.2f} MB")
print(f"Compressed size: {size_compressed_mb:.2f} MB")
print(f"Max absolute error: {error_max:.2e}")
print(f"Frobenius norm of error: {error_frob_norm:.2e}")
print(f"Relative Frobenius error: {error_rel_frob:.2e}")

Original size: 211.50 MB
Compressed size: 120.07 MB
Max absolute error: 3.20e-06
Frobenius norm of error: 2.86e-04
Relative Frobenius error: 3.77e-08
